# 🏦 Mini Project 3 — Agentic AI in FinTech
### Comparing Baseline, Single-Agent, and Multi-Agent Architectures


## 🎯 Learning Objectives

1. Understand how tool-calling (function calling) works with LLMs
2. Define your own tool schemas and wire them to real Python functions
3. Build a single-agent system and reason about its limitations
4. Design and implement a multi-agent system — architecture is your choice
5. Build an LLM-as-judge evaluator to score answers automatically
6. Analyse accuracy, hallucination rate, and latency across all architectures and models
7. Reflect critically: *when does added complexity actually pay off?*

---

## 📦 What You Are Given vs What You Build

| Component | Status |
|---|---|
| 5 working tool functions | ✅ Provided |
| JSON schemas for all 7 tools | ✅ Provided |
| `AgentResult` dataclass | ✅ Provided |
| `run_specialist_agent()` loop | ✅ Provided |
| 15 fixed benchmark questions | ✅ Provided |
| Evaluation runner + Excel writer | ✅ Provided |
| **Tool 6: `get_company_overview`** | ❌ You implement |
| **Tool 7: `get_tickers_by_sector`** | ❌ You implement |
| **Baseline** | ❌ You implement |
| **Single-agent system** | ❌ You design + implement |
| **Multi-agent system** | ❌ You design + implement |
| **Evaluator** | ❌ You design + implement |

---

## 🔑 Before You Start

**API keys needed:**
- OpenAI → https://platform.openai.com/api-keys  
- Alpha Vantage (free) → https://www.alphavantage.co/support/#api-key

**Data needed:**
- `sp500_companies.csv` → https://www.kaggle.com/datasets/andrewmvd/sp-500-stocks  
- Unzip and place next to this notebook

**Create a `.env` file** in the same folder (never commit this):
```
OPENAI_API_KEY=sk-proj-...
ALPHAVANTAGE_API_KEY=...
```


## Step 0 — Install & Import

In [ ]:
!pip install openai requests pandas yfinance python-dotenv openpyxl --quiet

In [ ]:
import os, json, time, sqlite3, requests, textwrap
import pandas as pd
import yfinance as yf
from dataclasses import dataclass, field
from dotenv import load_dotenv
from openai import OpenAI

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    ALPHAVANTAGE_API_KEY = userdata.get('AlphaVantage')
    print("Loaded keys from Google Colab Secrets")
except ImportError:
    load_dotenv()
    OPENAI_API_KEY = os.getenv("OpenAI_proj.3", "YOUR_KEY")
    ALPHAVANTAGE_API_KEY = os.getenv("ALPHA_proj3", "YOUR_KEY")
    print("Loaded keys from local .env file")

MODEL_SMALL  = "gpt-4o-mini"
MODEL_LARGE  = "gpt-4o"
ACTIVE_MODEL = MODEL_SMALL          # switch to MODEL_LARGE for the second run

client = OpenAI(api_key=OPENAI_API_KEY)
print(f"✅ Ready  |  active model: {ACTIVE_MODEL}")

Loaded keys from Google Colab Secrets
✅ Ready  |  active model: gpt-4o-mini


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive/')
os.chdir('/content/drive/My Drive/Colab Notebooks/TransformertoLLM/MP3')

Mounted at /content/drive/


## Step 1 — Build the Local Database

Run `create_local_database()` once after placing `sp500_companies.csv` next to this notebook.  
The function prints all distinct **sector** values — you will need these when implementing Tool 7.


In [ ]:
DB_PATH = "stocks.db"

def create_local_database(csv_path: str = "sp500_companies.csv"):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(
            f"'{csv_path}' not found.\n"
            "Download from: https://www.kaggle.com/datasets/andrewmvd/sp-500-stocks"
        )
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip().str.lower()
    df = df.rename(columns={
        "symbol":"ticker", "shortname":"company",
        "sector":"sector",  "industry":"industry",
        "exchange":"exchange", "marketcap":"market_cap_raw"
    })
    def cap_bucket(v):
        try:
            v = float(v)
            return "Large" if v >= 10_000_000_000 else "Mid" if v >= 2_000_000_000 else "Small"
        except: return "Unknown"
    df["market_cap"] = df["market_cap_raw"].apply(cap_bucket)
    df = (df.dropna(subset=["ticker","company"])
            .drop_duplicates(subset=["ticker"])
            [["ticker","company","sector","industry","market_cap","exchange"]])
    conn = sqlite3.connect(DB_PATH)
    df.to_sql("stocks", conn, if_exists="replace", index=False)
    conn.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_ticker ON stocks(ticker)")
    conn.commit()
    n = pd.read_sql_query("SELECT COUNT(*) AS n FROM stocks", conn).iloc[0]["n"]
    print(f"✅ {n} companies loaded into stocks.db")
    print("\nDistinct sector values stored in DB:")
    print(pd.read_sql_query("SELECT DISTINCT sector FROM stocks ORDER BY sector", conn).to_string(index=False))
    conn.close()

create_local_database()

✅ 502 companies loaded into stocks.db

Distinct sector values stored in DB:
                sector
       Basic Materials
Communication Services
     Consumer Cyclical
    Consumer Defensive
                Energy
    Financial Services
            Healthcare
           Industrials
           Real Estate
            Technology
             Utilities


## Step 2 — Tool Functions

### Provided Tools (5 of 7)

Read each function carefully — you need to understand their return shapes before writing agents.


In [ ]:
# ── Tool 1 ── Provided ────────────────────────────────────────
def get_price_performance(tickers: list, period: str = "1y") -> dict:
    """
    % price change for a list of tickers over a period.
    Valid periods: '1mo', '3mo', '6mo', 'ytd', '1y'
    Returns: { TICKER: {start_price, end_price, pct_change, period} }
    """
    results = {}
    for ticker in tickers:
        try:
            data = yf.download(ticker, period=period, progress=False, auto_adjust=True)
            if data.empty:
                results[ticker] = {"error": "No data — possibly delisted"}
                continue
            start = float(data["Close"].iloc[0].item())
            end   = float(data["Close"].iloc[-1].item())
            results[ticker] = {
                "start_price": round(start, 2),
                "end_price"  : round(end,   2),
                "pct_change" : round((end - start) / start * 100, 2),
                "period"     : period,
            }
        except Exception as e:
            results[ticker] = {"error": str(e)}
    return results

# ── Tool 2 ── Provided ────────────────────────────────────────
def get_market_status() -> dict:
    """Open / closed status for global stock exchanges."""
    return requests.get(
        f"https://www.alphavantage.co/query?function=MARKET_STATUS"
        f"&apikey={ALPHAVANTAGE_API_KEY}", timeout=10
    ).json()

# ── Tool 3 ── Provided ────────────────────────────────────────
def get_top_gainers_losers() -> dict:
    """Today's top gaining, top losing, and most active tickers."""
    return requests.get(
        f"https://www.alphavantage.co/query?function=TOP_GAINERS_LOSERS"
        f"&apikey={ALPHAVANTAGE_API_KEY}", timeout=10
    ).json()

# ── Tool 4 ── Provided ────────────────────────────────────────
def get_news_sentiment(ticker: str, limit: int = 5) -> dict:
    """
    Latest headlines + Bullish / Bearish / Neutral sentiment for a ticker.
    Returns: { ticker, articles: [{title, source, sentiment, score}] }
    """
    data = requests.get(
        f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT"
        f"&tickers={ticker}&limit={limit}&apikey={ALPHAVANTAGE_API_KEY}", timeout=10
    ).json()
    return {
        "ticker": ticker,
        "articles": [
            {
                "title"    : a.get("title"),
                "source"   : a.get("source"),
                "sentiment": a.get("overall_sentiment_label"),
                "score"    : a.get("overall_sentiment_score"),
            }
            for a in data.get("feed", [])[:limit]
        ],
    }

# ── Tool 5 ── Provided ────────────────────────────────────────
def query_local_db(sql: str) -> dict:
    """
    Run any SQL SELECT on stocks.db.
    Table 'stocks' columns: ticker, company, sector, industry, market_cap, exchange
    market_cap values: 'Large' | 'Mid' | 'Small'
    """
    try:
        conn = sqlite3.connect(DB_PATH)
        df   = pd.read_sql_query(sql, conn)
        conn.close()
        return {"columns": list(df.columns), "rows": df.to_dict(orient="records")}
    except Exception as e:
        return {"error": str(e)}

print("✅ 5 provided tools ready")

✅ 5 provided tools ready


---
## 🛠️ Task 1 — Implement the 2 Missing Tools (20 pts)

### Tool 6 — `get_company_overview`

Call the Alpha Vantage **OVERVIEW** endpoint for a single ticker.  
Docs: https://www.alphavantage.co/documentation/#company-overview  

Required return format:
```python
{
    "ticker"    : str,   # the input ticker
    "name"      : str,   # company full name
    "sector"    : str,
    "pe_ratio"  : str,   # field name in API: PERatio
    "eps"       : str,   # field name: EPS
    "market_cap": str,   # field name: MarketCapitalization
    "52w_high"  : str,   # field name: 52WeekHigh
    "52w_low"   : str,   # field name: 52WeekLow
}
```
If the API returns no `"Name"` key (rate-limited or invalid ticker), return:
```python
{"error": f"No overview data for {ticker}"}
```

---

### Tool 7 — `get_tickers_by_sector`

Query `stocks.db` for companies matching a sector **or** industry.

**Critical detail:** Look at the sector values printed by `create_local_database()`.  
The DB stores broad sectors in `sector` (e.g. `"Information Technology"`) and  
specific industries in `industry` (e.g. `"Semiconductors"`).  
A query for `"semiconductor"` must fall back to the `industry` column — otherwise it returns zero rows.

Required logic:
1. Try exact match on `sector` column (case-insensitive)  
2. If 0 results → try `LIKE '%sector%'` on the `industry` column  

Required return format:
```python
{
    "sector": str,          # the input search term
    "stocks": [
        {"ticker": str, "company": str, "industry": str},
        ...
    ]
}
```


In [ ]:
# ── Tool 6 — YOUR IMPLEMENTATION ─────────────────────────────
def get_company_overview(ticker: str) -> dict:
    url = f"https://www.alphavantage.co/query?function=OVERVIEW&symbol={ticker}&apikey={ALPHAVANTAGE_API_KEY}"
    try:
        data = requests.get(url, timeout=10).json()
        if "Name" not in data:
            return {"error": f"No overview data for {ticker}"}
        return {
            "ticker": ticker,
            "name": data.get("Name", ""),
            "sector": data.get("Sector", ""),
            "pe_ratio": data.get("PERatio", ""),
            "eps": data.get("EPS", ""),
            "market_cap": data.get("MarketCapitalization", ""),
            "52w_high": data.get("52WeekHigh", ""),
            "52w_low": data.get("52WeekLow", "")
        }
    except Exception as e:
        return {"error": str(e)}


# ── Tool 7 — YOUR IMPLEMENTATION ─────────────────────────────
def get_tickers_by_sector(sector: str) -> dict:
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT ticker, company, industry FROM stocks WHERE LOWER(sector) = LOWER(?)", conn, params=(sector,))
    if df.empty:
        pattern = f"%{sector}%"
        df = pd.read_sql_query("SELECT ticker, company, industry FROM stocks WHERE industry LIKE ?", conn, params=(pattern,))
    conn.close()
    return {
        "sector": sector,
        "stocks": df.to_dict(orient="records")
    }


# ── Automated tests — all assertions must pass ────────────────
print("=== Tool 6 tests ===")
aapl = get_company_overview("AAPL")
assert "pe_ratio" in aapl,             "Missing pe_ratio key"
assert aapl.get("ticker") == "AAPL",   "ticker key wrong"
assert aapl.get("name"),               "name should not be empty"
print(f"  AAPL P/E: {aapl['pe_ratio']} ✅")

bad = get_company_overview("INVALIDTICKER999")
assert "error" in bad,                 "Invalid ticker should return error key"
print(f"  Invalid ticker handled correctly ✅")

print("\n=== Tool 7 tests ===")
tech = get_tickers_by_sector("Information Technology")
assert len(tech["stocks"]) > 0,        "Should find IT stocks (exact sector match)"
print(f"  'Information Technology' → {len(tech['stocks'])} stocks ✅")

semi = get_tickers_by_sector("semiconductor")
assert len(semi["stocks"]) > 0,        "Should find via industry fallback (LIKE match)"
print(f"  'semiconductor' (industry fallback) → {len(semi['stocks'])} stocks ✅")

print("\n✅ All tool tests passed")

=== Tool 6 tests ===
  AAPL P/E: 32.37 ✅
  Invalid ticker handled correctly ✅

=== Tool 7 tests ===
  'Information Technology' → 11 stocks ✅
  'semiconductor' (industry fallback) → 18 stocks ✅

✅ All tool tests passed


## Step 3 — Tool Schemas (Provided)

Schemas tell the LLM what tools exist, what they do, and what arguments they take.  
You do not modify these — but you will reference the schema lists when building your agents.

**Key concept:** You can give different agents different *subsets* of schemas.  
A specialist that only sees 2–3 relevant schemas makes fewer wrong tool choices  
than one that sees all 7.


In [ ]:
def _s(name, desc, props, req):
    return {"type":"function","function":{
        "name":name,"description":desc,
        "parameters":{"type":"object","properties":props,"required":req}}}

SCHEMA_TICKERS  = _s("get_tickers_by_sector",
    "Return all stocks in a sector or industry from the local database. "
    "Use broad sector names ('Information Technology', 'Energy') or sub-sectors ('semiconductor', 'insurance').",
    {"sector":{"type":"string","description":"Sector or industry name"}}, ["sector"])

SCHEMA_PRICE    = _s("get_price_performance",
    "Get % price change for a list of tickers over a time period. "
    "Periods: '1mo','3mo','6mo','ytd','1y'.",
    {"tickers":{"type":"array","items":{"type":"string"}},
     "period":{"type":"string","default":"1y"}}, ["tickers"])

SCHEMA_OVERVIEW = _s("get_company_overview",
    "Get fundamentals for one stock: P/E ratio, EPS, market cap, 52-week high and low.",
    {"ticker":{"type":"string","description":"Ticker symbol e.g. 'AAPL'"}}, ["ticker"])

SCHEMA_STATUS   = _s("get_market_status",
    "Check whether global stock exchanges are currently open or closed.", {}, [])

SCHEMA_MOVERS   = _s("get_top_gainers_losers",
    "Get today's top gaining, top losing, and most actively traded stocks.", {}, [])

SCHEMA_NEWS     = _s("get_news_sentiment",
    "Get latest news headlines and Bullish/Bearish/Neutral sentiment scores for a stock.",
    {"ticker":{"type":"string"},"limit":{"type":"integer","default":5}}, ["ticker"])

SCHEMA_SQL      = _s("query_local_db",
    "Run a SQL SELECT on stocks.db. "
    "Table 'stocks': ticker, company, sector, industry, market_cap (Large/Mid/Small), exchange.",
    {"sql":{"type":"string","description":"A valid SQL SELECT statement"}}, ["sql"])

# All 7 schemas in one list — used by single agent
ALL_SCHEMAS = [SCHEMA_TICKERS, SCHEMA_PRICE, SCHEMA_OVERVIEW,
               SCHEMA_STATUS, SCHEMA_MOVERS, SCHEMA_NEWS, SCHEMA_SQL]

# Dispatch map — maps the tool name string the LLM returns → the Python function to call
ALL_TOOL_FUNCTIONS = {
    "get_tickers_by_sector" : get_tickers_by_sector,
    "get_price_performance"  : get_price_performance,
    "get_company_overview"   : get_company_overview,
    "get_market_status"      : get_market_status,
    "get_top_gainers_losers" : get_top_gainers_losers,
    "get_news_sentiment"     : get_news_sentiment,
    "query_local_db"         : query_local_db,
}

print("✅ Schemas ready")
print(f"   Tools available: {list(ALL_TOOL_FUNCTIONS.keys())}")

✅ Schemas ready
   Tools available: ['get_tickers_by_sector', 'get_price_performance', 'get_company_overview', 'get_market_status', 'get_top_gainers_losers', 'get_news_sentiment', 'query_local_db']


## Step 4 — AgentResult and Base Runner (Provided)

`AgentResult` is the standard return type for every agent — baseline, single, and all multi-agent specialists.  
`run_specialist_agent` is the reusable tool-call loop that every agent uses.  
Study this loop carefully before writing your own agents — it is what connects the LLM's tool requests to your Python functions.


In [ ]:
@dataclass
class AgentResult:
    agent_name   : str
    answer       : str
    tools_called : list  = field(default_factory=list)   # tool names in order called
    raw_data     : dict  = field(default_factory=dict)   # tool name → raw tool output
    confidence   : float = 0.0                           # set by evaluator / critic
    issues_found : list  = field(default_factory=list)   # set by evaluator / critic
    reasoning    : str   = ""

    def summary(self):
        print(f"\n{'─'*54}")
        print(f"Agent      : {self.agent_name}")
        print(f"Tools used : {', '.join(self.tools_called) or 'none'}")
        print(f"Confidence : {self.confidence:.0%}")
        if self.issues_found:
            print(f"Issues     : {'; '.join(self.issues_found)}")
        print(f"Answer     :\n{textwrap.indent(self.answer[:500], '  ')}")

print("✅ AgentResult defined")

✅ AgentResult defined


In [ ]:
def run_specialist_agent(
    agent_name   : str,
    system_prompt: str,
    task         : str,
    tool_schemas : list,
    max_iters    : int  = 8,
    verbose      : bool = True,
) -> AgentResult:
    """
    Core agentic loop used by every agent in this project.

    How it works:
      1. Sends system_prompt + task to the LLM
      2. If the LLM requests a tool call → looks up the function in ALL_TOOL_FUNCTIONS,
         executes it, appends the result to the message history, loops back to step 1
      3. When the LLM produces a response with no tool calls → returns an AgentResult

    Parameters
    ----------
    agent_name    : display name for logging
    system_prompt : the agent's persona, rules, and focus area
    task          : the specific question or sub-task for this agent
    tool_schemas  : list of schema dicts this agent is allowed to use
                    (pass [] for no tools — used by baseline)
    max_iters     : hard cap on iterations to prevent infinite loops
    verbose       : print each tool call as it happens
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": task}
    ]
    tools_called = []
    raw_data = {}

    for _ in range(max_iters):
        response = client.chat.completions.create(
            model=ACTIVE_MODEL,
            messages=messages,
            tools=tool_schemas if tool_schemas else None,
            temperature=0.0
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return AgentResult(
                agent_name=agent_name,
                answer=msg.content or "",
                tools_called=tools_called,
                raw_data=raw_data
            )

        messages.append(msg)
        for tool_call in msg.tool_calls:
            func_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            if verbose:
                print(f"[{agent_name}] Calling {func_name} with {args}")

            tools_called.append(func_name)

            if func_name in ALL_TOOL_FUNCTIONS:
                try:
                    res = ALL_TOOL_FUNCTIONS[func_name](**args)
                except Exception as e:
                    res = {"error": str(e)}
            else:
                res = {"error": f"Unknown function: {func_name}"}

            raw_data[func_name] = res
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": func_name,
                "content": json.dumps(res)
            })

    return AgentResult(
        agent_name=agent_name,
        answer="Error: max_iters reached",
        tools_called=tools_called,
        raw_data=raw_data
    )
print("✅ run_specialist_agent ready")

✅ run_specialist_agent ready


---
## 🛠️ Task 2 — Implement the Baseline (10 pts)

The baseline is a **single LLM call with no tools**. The model answers entirely from its training knowledge.

This is your **control condition**. Every architecture you build should be compared against it.  
If agents don't outperform the baseline, there's no justification for the extra complexity.

**Requirements:**
- One call to `client.chat.completions.create()` — no `tools` argument
- Return `AgentResult(agent_name="Baseline", answer=..., tools_called=[])`
- Use a sensible system prompt that encourages the model to be honest about uncertainty

**Grading checks:**
- `tools_called` must be `[]`
- Answer must not be empty


In [ ]:
def run_baseline(question: str, verbose: bool = True) -> AgentResult:
    response = client.chat.completions.create(
        model=ACTIVE_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful financial AI assistant. Answer the user's question directly based on your training knowledge. Do not use tools. If you are uncertain about exact numbers, state that you are making an estimate."},
            {"role": "user", "content": question}
        ],
        temperature=0.0
    )
    answer = response.choices[0].message.content
    return AgentResult(
        agent_name="Baseline",
        answer=answer,
        tools_called=[]
    )
    # Implement a single LLM call with no tools.
    # Use run_specialist_agent() with an empty tool_schemas list — or make the call directly.
    # Return an AgentResult with agent_name="Baseline" and tools_called=[].

# Quick test
bl = run_baseline("What is Apple's approximate P/E ratio?", verbose=True)
assert bl.tools_called == [], "Baseline must not call any tools"
assert bl.answer, "Answer must not be empty"
bl.summary()


──────────────────────────────────────────────────────
Agent      : Baseline
Tools used : none
Confidence : 0%
Answer     :
  As of my last knowledge update in October 2023, Apple's approximate price-to-earnings (P/E) ratio was around 25 to 30. However, P/E ratios can fluctuate frequently based on stock price changes and earnings reports, so I recommend checking a financial news website or stock market app for the most current figure.


---
## 🛠️ Task 3 — Design and Implement the Single Agent (20 pts)

A single agent is **one LLM with access to all 7 tools**. Everything — planning which tools to call, executing them, and synthesising the answer — happens in one context window.

You decide how to build it. The guidance below is a starting point, not a recipe.

---

### Things to think about when writing your system prompt

**Role and scope** — what is this agent's job? Being specific helps the model stay focused.

**Accuracy rules** — how should the agent behave when a tool returns an error or empty data?  
This is critical: an agent with no rules tends to fabricate plausible-looking numbers when the API fails.

**Multi-step reasoning** — some questions require chaining tools. For example:  
*"Which energy stocks grew the most this year?"* requires first looking up energy tickers,  
then fetching price data for each one. Without explicit guidance, single agents often  
skip the lookup step and guess tickers instead.

**Tool selection** — the agent sees all 7 tools. Giving it rules about *when* to use each  
(e.g. "use `query_local_db` with SQL when you need to filter by sector or market cap")  
reduces wrong tool choices.

---

### Recommended structure

```python
SINGLE_AGENT_PROMPT = """
<your system prompt here>
"""

def run_single_agent(question: str, verbose: bool = True) -> AgentResult:
    # Call run_specialist_agent() with:
    #   agent_name    = "Single Agent"
    #   system_prompt = SINGLE_AGENT_PROMPT
    #   task          = question
    #   tool_schemas  = ALL_SCHEMAS   (all 7)
    #   max_iters     = 10
    # Return the AgentResult directly.
    pass
```

---

### Test before you move on

Run the three cells below and check the results make sense before building the multi-agent system.


In [ ]:
# ── YOUR SINGLE AGENT IMPLEMENTATION ─────────────────────────
# Write your system prompt and run_single_agent() function here.
# Comments above explain what to think about — the implementation is yours.

SINGLE_AGENT_PROMPT = """
You are an expert financial AI assistant equipped with tools to fetch live stock data and interact with a local database.
Your goal is to answer the user's questions truthfully and accurately using ONLY the data provided by your tools.

Follow these STRICT RULES:
1. NEVER fabricate or guess any data (prices, P/E ratios, tickers, news sentiment, etc.). If a tool returns an error or empty data, state that clearly instead of inventing a number.
2. If the user asks about a SECTOR or INDUSTRY, you MUST ALWAYS use the `get_tickers_by_sector` tool FIRST to find the relevant tickers in the database. DO NOT guess which tickers belong to a sector.
3. If the user asks a multi-step question (e.g. "Which energy stocks grew the most?"), you MUST:
   a. Use `get_tickers_by_sector` to get the tickers.
   b. Extract the ticker symbols from the result.
   c. Pass those ticker symbols into `get_price_performance` or `get_company_overview` as required by the question.
4. For filtering by market cap or executing generic DB filters, you can use `query_local_db` if needed.
5. Provide your final answer clearly, confirming the data used to reach your conclusion.
"""

def run_single_agent(question: str, verbose: bool = True) -> AgentResult:
    return run_specialist_agent(
        agent_name="Single Agent",
        system_prompt=SINGLE_AGENT_PROMPT,
        task=question,
        tool_schemas=ALL_SCHEMAS,
        max_iters=15,
        verbose=verbose
    )



In [ ]:
BENCHMARK_QUESTIONS = [
    # ── EASY ──────────────────────────────────────────────────────────────
    {"id":"Q01","complexity":"easy","category":"sector_lookup",
     "question":"List all semiconductor companies in the database.",
     "expected":"Should return company names and tickers for semiconductor stocks from the local DB. "
                "Tickers include NVDA, AMD, INTC, QCOM, AVGO, TXN, ADI, MU and others."},
    {"id":"Q02","complexity":"easy","category":"market_status",
     "question":"Are the US stock markets open right now?",
     "expected":"Should return the current open/closed status for NYSE and NASDAQ "
                "with their trading hours."},
    {"id":"Q03","complexity":"easy","category":"fundamentals",
     "question":"What is the P/E ratio of Apple (AAPL)?",
     "expected":"Should return AAPL P/E ratio as a single numeric value fetched from Alpha Vantage."},
    {"id":"Q04","complexity":"easy","category":"sentiment",
     "question":"What is the latest news sentiment for Microsoft (MSFT)?",
     "expected":"Should return 3–5 recent MSFT headlines with Bullish/Bearish/Neutral labels and scores."},
    {"id":"Q05","complexity":"easy","category":"price",
     "question":"What is NVIDIA's stock price performance over the last month?",
     "expected":"Should return NVDA start price, end price, and % change for the 1-month period."},

    # ── MEDIUM ─────────────────────────────────────────────────────────────
    {"id":"Q06","complexity":"medium","category":"price_comparison",
     "question":"Compare the 1-year price performance of AAPL, MSFT, and GOOGL. Which grew the most?",
     "expected":"Should fetch 1y performance for all 3 tickers, return % change for each, "
                "and identify the highest performer."},
    {"id":"Q07","complexity":"medium","category":"fundamentals",
     "question":"Compare the P/E ratios of AAPL, MSFT, and NVDA. Which looks most expensive?",
     "expected":"Should return P/E ratios for all 3 tickers and identify which has the highest P/E."},
    {"id":"Q08","complexity":"medium","category":"sector_price",
     "question":"Which energy stocks in the database had the best 6-month performance?",
     "expected":"Should query the DB for energy sector tickers, fetch 6-month price performance "
                "for each, and return them ranked by % change."},
    {"id":"Q09","complexity":"medium","category":"sentiment",
     "question":"What is the news sentiment for Tesla (TSLA) and how has its stock moved this month?",
     "expected":"Should return TSLA news sentiment (label + score) AND 1-month price % change "
                "from two separate tool calls."},
    {"id":"Q10","complexity":"medium","category":"fundamentals",
     "question":"What are the 52-week high and low for JPMorgan (JPM) and Goldman Sachs (GS)?",
     "expected":"Should return 52-week high and low for both JPM and GS fetched from Alpha Vantage."},

    # ── HARD ───────────────────────────────────────────────────────────────
    {"id":"Q11","complexity":"hard","category":"multi_condition",
     "question":"Which tech stocks dropped this month but grew this year? Return the top 3.",
     "expected":"Should get tech tickers from DB, fetch both 1-month and year-to-date performance, "
                "filter for negative 1-month AND positive YTD, return top 3 by yearly growth with "
                "exact percentages. Results must satisfy both conditions simultaneously."},
    {"id":"Q12","complexity":"hard","category":"multi_condition",
     "question":"Which large-cap technology stocks on NASDAQ have grown more than 20% this year?",
     "expected":"Should query DB for large-cap NASDAQ tech stocks, fetch YTD performance, "
                "filter for >20% growth, and return matching tickers with exact % change."},
    {"id":"Q13","complexity":"hard","category":"cross_domain",
     "question":"For the top 3 semiconductor stocks by 1-year return, what are their P/E ratios "
                "and current news sentiment?",
     "expected":"Should find semiconductor tickers in DB, rank by 1-year return to find top 3, "
                "then fetch P/E ratio AND news sentiment for each — requiring three separate "
                "data domains (price, fundamentals, sentiment)."},
    {"id":"Q14","complexity":"hard","category":"cross_domain",
     "question":"Compare the market cap, P/E ratio, and 1-year stock performance of JPM, GS, and BAC.",
     "expected":"Should return market cap, P/E, and 1-year % change for all 3 tickers, "
                "combining Alpha Vantage fundamentals and yfinance price data."},
    {"id":"Q15","complexity":"hard","category":"multi_condition",
     "question":"Which finance sector stocks are trading closer to their 52-week low than their "
                "52-week high? Return the news sentiment for each.",
     "expected":"Should get finance sector tickers from DB, fetch 52-week high and low for each, "
                "compute proximity to the low, then fetch news sentiment for qualifying stocks."},
]

print("--- Running Single Agent on 5 Benchmark Questions ---")
for q in BENCHMARK_QUESTIONS[:5]:
    print(f"\nQ: {q['question']}")
    res = run_single_agent(q['question'])
    if res:
        res.summary()


--- Running Single Agent on 5 Benchmark Questions ---

Q: List all semiconductor companies in the database.
[Single Agent] Calling get_tickers_by_sector with {'sector': 'semiconductor'}

──────────────────────────────────────────────────────
Agent      : Single Agent
Tools used : get_tickers_by_sector
Confidence : 0%
Answer     :
  Here is a list of semiconductor companies in the database:

  1. **NVIDIA Corporation (NVDA)**
  2. **Broadcom Inc. (AVGO)**
  3. **Advanced Micro Devices, Inc. (AMD)**
  4. **Texas Instruments Incorporated (TXN)**
  5. **QUALCOMM Incorporated (QCOM)**
  6. **Applied Materials, Inc. (AMAT)**
  7. **Analog Devices, Inc. (ADI)**
  8. **Micron Technology, Inc. (MU)**
  9. **Lam Research Corporation (LRCX)**
  10. **Intel Corporation (INTC)**
  11. **KLA Corporation (KLAC)**
  12. **NXP Semiconductors N.V. (NXPI)**
  13. **M

Q: Are the US stock markets open right now?
[Single Agent] Calling get_market_status with {}

────────────────────────────────────────────

---
## 🛠️ Task 4 — Design and Implement a Multi-Agent System (25 pts)

You must build a multi-agent system that answers the same 15 questions.  
**The architecture is your choice.** Experiment, compare, and justify your decision in the reflections.

---

### Three architectures to consider

**Orchestrator + Specialists + Critic**
```
User Question
     │
 Orchestrator  ← reads question, writes a plan, delegates sub-tasks
     │
 ┌───┼───┐
 Ag1 Ag2 Ag3   ← each handles one domain, sees a subset of tools
 └───┼───┘
  Critic        ← fact-checks each agent's answer vs its raw tool data
     │
 Synthesizer    ← merges verified answers into one final response
```
*Good for:* complex cross-domain questions  
*Tradeoff:* most latency, most API calls, but most transparency

---

**Sequential Pipeline**
```
User Question → Agent1 → Agent2 → Agent3 → Final Answer
```
Each agent receives the previous agent's output as context.  
*Good for:* questions with a natural order (find tickers → get prices → summarise)  
*Tradeoff:* errors propagate — a wrong ticker in step 1 breaks all later steps

---

**Parallel Specialists + Aggregator**
```
User Question
  ├── Price Agent       ─┐
  ├── Fundamentals Agent  ├─→ Aggregator → Final Answer
  └── Sentiment Agent   ─┘
```
All specialists run, aggregator merges results.  
*Good for:* speed (can use `ThreadPoolExecutor` to run in parallel)  
*Tradeoff:* no cross-checking between agents

---

### Suggested tool subsets per specialist
These are starting points — adjust based on your design:

```python
MARKET_TOOLS      = [SCHEMA_TICKERS, SCHEMA_PRICE, SCHEMA_STATUS, SCHEMA_MOVERS]
FUNDAMENTAL_TOOLS = [SCHEMA_OVERVIEW, SCHEMA_SQL, SCHEMA_TICKERS]
SENTIMENT_TOOLS   = [SCHEMA_NEWS, SCHEMA_SQL]
```

Giving each specialist only its relevant schemas reduces wrong tool choices.

---

### Required return contract

Whatever architecture you choose, `run_multi_agent()` must return this dict  
(the evaluation runner reads these fields):

```python
{
    "final_answer"  : str,         # the answer shown to the user
    "agent_results" : list,        # list[AgentResult] — one per specialist that ran
    "elapsed_sec"   : float,       # total wall clock time
    "architecture"  : str,         # short name e.g. "orchestrator-critic", "pipeline", "parallel"
}
```

The evaluation runner extracts from `agent_results`:
- `tools_called` (all tools used across specialists)
- `confidence` (averaged across specialists)
- `issues_found` (all issues concatenated)

---

### Document your design choices in document

The grading for this task includes your reasoning — explain in document:
- Which architecture you chose and why
- What you tried that didn't work
- How you divided tools between specialists
- What your verification/confidence mechanism does


In [ ]:
# ── YOUR MULTI-AGENT IMPLEMENTATION ──────────────────────────
# See Reflection Question for the detailed breakdown of the multi-agent architecture
# Architecture chosen: [describe your choice here]
# Reason: [explain why you chose this over the alternatives]
#
# Specialist breakdown:
#   Agent 1 — [name, domain, tool subset]
#   Agent 2 — [name, domain, tool subset]
#   Agent N — [name, domain, tool subset]
#
# Verification mechanism: [how does your system check answer quality?]
#
### YOUR CODE HERE

import json
import time
from typing import List, Dict, Any

# ==========================================
# OPENAI HELPER FOR MULTI-AGENT
# ==========================================

def get_chat_completion(messages, temperature=0.0):
    """Helper to call OpenAI using the notebook's existing client and model."""
    import __main__
    client = getattr(__main__, 'client', None)
    model = getattr(__main__, 'ACTIVE_MODEL', "gpt-4o-mini")

    if client is None:
        import os
        from openai import OpenAI
        client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    return client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )

def get_all_schemas():
    import __main__
    return getattr(__main__, 'ALL_SCHEMAS', [])

# ==========================================
# 1. DATABASE AGENT
# ==========================================

DATABASE_AGENT_PROMPT = """
You are a Database Specialist Agent.
Your role is to query the local database to find stock tickers based on sectors or industries.
Use the `get_tickers_by_sector` tool to find requested companies.
Only return information based on the data retrieved from the database. Do not hallucinate tickers.
"""

def run_database_agent(task: str, schemas: list, verbose: bool = True):
    import __main__
    run_sp = getattr(__main__, 'run_specialist_agent', None)
    if not run_sp: raise ImportError("Missing run_specialist_agent in global scope")

    valid_tools = ["get_tickers_by_sector", "query_local_db"]
    agent_schemas = [s for s in schemas if s.get("function", {}).get("name") in valid_tools]
    if not agent_schemas and schemas: agent_schemas = schemas

    return run_sp(
        agent_name="Database Agent",
        system_prompt=DATABASE_AGENT_PROMPT,
        task=task,
        tool_schemas=agent_schemas,
        max_iters=5,
        verbose=verbose
    )

# ==========================================
# 2. FUNDAMENTALS AGENT
# ==========================================

FUNDAMENTALS_AGENT_PROMPT = """
You are a Fundamentals Specialist Agent.
Your role is to retrieve static or point-in-time company metrics, such as P/E ratio, EPS, market cap, and 52-week high/low.
Use the `get_company_overview` tool to fetch this data for specific stock tickers.
Never invent financial numbers. If the data is missing or returns an error, state that clearly.
"""

def run_fundamentals_agent(task: str, schemas: list, verbose: bool = True):
    import __main__
    run_sp = getattr(__main__, 'run_specialist_agent', None)

    valid_tools = ["get_company_overview"]
    agent_schemas = [s for s in schemas if s.get("function", {}).get("name") in valid_tools]
    if not agent_schemas and schemas: agent_schemas = schemas

    return run_sp(
        agent_name="Fundamentals Agent",
        system_prompt=FUNDAMENTALS_AGENT_PROMPT,
        task=task,
        tool_schemas=agent_schemas,
        max_iters=5,
        verbose=verbose
    )

# ==========================================
# 3. TECHNICAL/PRICE AGENT
# ==========================================

TECHNICAL_AGENT_PROMPT = """
You are a Technical and Price Specialist Agent.
Your role is to handle historical price performance and check current market open/close status.
Use the `get_price_performance` tool to evaluate how a stock or list of stocks performed over time.
Use the `get_market_status` tool to check if the market is open.
Do not hallucinate percentage changes or prices. Base all answers strictly on tool outputs.
"""

def run_technical_agent(task: str, schemas: list, verbose: bool = True):
    import __main__
    run_sp = getattr(__main__, 'run_specialist_agent', None)

    valid_tools = ["get_price_performance", "get_market_status"]
    agent_schemas = [s for s in schemas if s.get("function", {}).get("name") in valid_tools]
    if not agent_schemas and schemas: agent_schemas = schemas

    return run_sp(
        agent_name="Technical/Price Agent",
        system_prompt=TECHNICAL_AGENT_PROMPT,
        task=task,
        tool_schemas=agent_schemas,
        max_iters=5,
        verbose=verbose
    )

# ==========================================
# 4. SENTIMENT AGENT
# ==========================================

SENTIMENT_AGENT_PROMPT = """
You are a Sentiment Specialist Agent.
Your role is to retrieve and analyze recent news headlines and sentiment for specific tickers.
Use the `get_news_sentiment` tool to fetch news articles and their sentiment scores.
Summarize the sentiment accurately without fabricating news stories.
"""

def run_sentiment_agent(task: str, schemas: list, verbose: bool = True):
    import __main__
    run_sp = getattr(__main__, 'run_specialist_agent', None)

    valid_tools = ["get_news_sentiment"]
    agent_schemas = [s for s in schemas if s.get("function", {}).get("name") in valid_tools]
    if not agent_schemas and schemas: agent_schemas = schemas

    return run_sp(
        agent_name="Sentiment Agent",
        system_prompt=SENTIMENT_AGENT_PROMPT,
        task=task,
        tool_schemas=agent_schemas,
        max_iters=5,
        verbose=verbose
    )

# ==========================================
# ORCHESTRATOR
# ==========================================

ORCHESTRATOR_PROMPT = """
You are the Orchestrator Agent.
Your job is to read the user's financial question and create a step-by-step sequential plan to answer it.
You have access to the following specialist agents:
- Database Agent: Finds stock tickers based on sectors or industries from the local database.
- Fundamentals Agent: Gets company metrics (P/E, EPS, market cap, 52-week high/low).
- Technical/Price Agent: Gets historical price performance and market status.
- Sentiment Agent: Gets recent news sentiment.

Output your plan as a valid JSON LIST of objects, where each object has:
- "agent": The exact name of the agent to use (must be one of the four above).
- "task": The specific instruction for that agent.

Since agents are executed sequentially, later agents can rely on data found by earlier agents (e.g., getting tickers first, then getting price for those tickers). So, if a question involves multiple domains, ensure the agents are ordered logically.
IMPORTANT: Return ONLY raw JSON list, without markdown fences.
Example:
[
  {"agent": "Database Agent", "task": "Find tickers for the energy sector."},
  {"agent": "Technical/Price Agent", "task": "Fetch 6-month price performance for the tickers found."}
]
"""

def get_orchestrator_plan(question: str, verbose: bool = True) -> list:
    response = get_chat_completion([
        {"role": "system", "content": ORCHESTRATOR_PROMPT},
        {"role": "user", "content": question}
    ])
    content = response.choices[0].message.content.strip()

    if content.startswith("```json"):
        content = content[7:-3].strip()
    elif content.startswith("```"):
        content = content[3:-3].strip()

    try:
        return json.loads(content)
    except Exception as e:
        if verbose: print(f"Failed to parse orchestrator plan: {e}. Running all agents individually.")
        return [
            {"agent": "Database Agent", "task": question},
            {"agent": "Technical/Price Agent", "task": question},
            {"agent": "Fundamentals Agent", "task": question},
            {"agent": "Sentiment Agent", "task": question}
        ]

# ==========================================
# CRITIC
# ==========================================

CRITIC_PROMPT = """
You are the Critic Agent.
Your job is to fact-check a specialist agent's answer against the raw data it retrieved.
Input format:
Agent Answer: <answer>
Raw Data: <JSON dump of tool outputs>

Tasks:
1. Verify that every number, price, sentiment label or definitive claim in the answer is supported by the raw data.
2. Identify any hallucinations or unsupported numbers.
3. Determine a confidence score from 0.0 (completely hallucinated/unsupported) to 1.0 (perfectly supported or clearly explains why data was unavailable).
4. List any specific issues found.

Output your review as a valid JSON object strictly matching this format:
{
  "confidence": 0.95,
  "issues_found": ["issue 1 here..."] // empty list if none
}
IMPORTANT: Return ONLY raw JSON, without markdown fences.
"""

def run_critic(agent_result, verbose: bool = True):
    content = f"Agent Answer: {agent_result.answer}\nRaw Data: {json.dumps(agent_result.raw_data)}"
    response = get_chat_completion([
        {"role": "system", "content": CRITIC_PROMPT},
        {"role": "user", "content": content}
    ])
    res_text = response.choices[0].message.content.strip()

    if res_text.startswith("```json"):
        res_text = res_text[7:-3].strip()
    elif res_text.startswith("```"):
        res_text = res_text[3:-3].strip()

    try:
        parsed = json.loads(res_text)
        agent_result.confidence = float(parsed.get("confidence", 0.0))
        agent_result.issues_found = parsed.get("issues_found", [])
    except Exception as e:
        if verbose: print(f"Critic parse error: {e}")
        agent_result.confidence = 0.5
        agent_result.issues_found = ["Critic evaluation failed to parse"]

# ==========================================
# SYNTHESIZER
# ==========================================

SYNTHESIZER_PROMPT = """
You are the Synthesizer Agent.
Your job is to answer the user's original question using ONLY the provided verified findings from specialist agents.
1. Formulate a cohesive, clear final answer.
2. Directly address the user's question.
3. If the findings lack information, clearly state what couldn't be found.
4. Do not invent details beyond what the specialists provided.
"""

def run_synthesizer(question: str, agent_results: list, verbose: bool = True) -> str:
    findings = []
    for r in agent_results:
        conf = getattr(r, 'confidence', 0.0)
        iss = getattr(r, 'issues_found', [])
        findings.append(
            f"--- {r.agent_name} (Confidence: {conf:.2f}) ---\n"
            f"Answer: {r.answer}\n"
            f"Issues: {', '.join(iss) if iss else 'None'}"
        )

    content = f"Original Question: {question}\n\nSpecialist Findings:\n" + "\n\n".join(findings)

    response = get_chat_completion([
        {"role": "system", "content": SYNTHESIZER_PROMPT},
        {"role": "user", "content": content}
    ])
    return response.choices[0].message.content

# ==========================================
# MULTI-AGENT RUNNER
# ==========================================

def run_multi_agent(question: str, verbose: bool = True) -> dict:
    start_time = time.time()

    schemas = get_all_schemas()

    if verbose: print(f"[Multi-Agent] Generating orchestration plan...")
    plan = get_orchestrator_plan(question, verbose=verbose)

    agent_results = []
    accumulated_context = []

    agent_funcs = {
        "Database Agent": run_database_agent,
        "Fundamentals Agent": run_fundamentals_agent,
        "Technical/Price Agent": run_technical_agent,
        "Sentiment Agent": run_sentiment_agent
    }

    for step in plan:
        agent_name = step.get("agent")
        sub_task = step.get("task")

        if agent_name not in agent_funcs:
            continue

        full_task = f"Original Question: {question}\n\nYour Specific Task: {sub_task}"
        if accumulated_context:
            full_task += "\n\nContext from previous agents (use this data if needed!):\n" + "\n".join(accumulated_context)

        if verbose: print(f"--- Running {agent_name} ---")

        # Execute specialist
        res = agent_funcs[agent_name](full_task, schemas=schemas, verbose=verbose)

        if verbose: print(f"--- Critic Evaluating {agent_name} ---")
        run_critic(res, verbose=verbose)

        agent_results.append(res)
        accumulated_context.append(f"[{agent_name} output]: {res.answer}")

    if verbose: print("--- Synthesizer Generating Final Answer ---")
    final_answer = run_synthesizer(question, agent_results, verbose=verbose)

    elapsed = time.time() - start_time

    return {
        "final_answer": final_answer,
        "agent_results": agent_results,
        "elapsed_sec": elapsed,
        "architecture": "orchestrator-critic"
    }



In [ ]:
# Test 1 — check return contract
out1 = run_multi_agent("What is the P/E ratio of Apple (AAPL)?")
assert "final_answer"   in out1, "Missing final_answer key"
assert "agent_results"  in out1, "Missing agent_results key"
assert "elapsed_sec"    in out1, "Missing elapsed_sec key"
assert "architecture"   in out1, "Missing architecture key"
print(f"Architecture : {out1['architecture']}")
print(f"Agents ran   : {[r.agent_name for r in out1['agent_results']]}")
print(f"Answer       : {out1['final_answer'][:200]}")

[Multi-Agent] Generating orchestration plan...
--- Running Fundamentals Agent ---
[Fundamentals Agent] Calling get_company_overview with {'ticker': 'AAPL'}
--- Critic Evaluating Fundamentals Agent ---
--- Synthesizer Generating Final Answer ---
Architecture : orchestrator-critic
Agents ran   : ['Fundamentals Agent']
Answer       : The P/E ratio for Apple Inc. (AAPL) is 32.37.


In [ ]:
# Test 2 — cross-domain hard question
out2 = run_multi_agent("For the top 3 semiconductor stocks by 1-year return, what are their P/E ratios and current news sentiment?")
print(f"Architecture : {out2['architecture']}")
print(f"Agents ran   : {[r.agent_name for r in out2['agent_results']]}")
print(f"Tools used   : {[t for r in out2['agent_results'] for t in r.tools_called]}")
print(f"\nAnswer:\n{out2['final_answer'][:400]}")

[Multi-Agent] Generating orchestration plan...
--- Running Database Agent ---
[Database Agent] Calling get_tickers_by_sector with {'sector': 'semiconductor'}
--- Critic Evaluating Database Agent ---
--- Running Technical/Price Agent ---
[Technical/Price Agent] Calling get_price_performance with {'tickers': ['NVDA', 'AVGO', 'AMD', 'TXN', 'QCOM', 'AMAT', 'ADI', 'MU', 'LRCX', 'INTC', 'KLAC', 'NXPI', 'MCHP', 'MPWR', 'ON', 'TER', 'SWKS', 'QRVO'], 'period': '1y'}
[Technical/Price Agent] Calling get_price_performance with {'tickers': ['MU', 'AMAT', 'LRCX'], 'period': '1y'}
[Technical/Price Agent] Calling get_market_status with {}
--- Critic Evaluating Technical/Price Agent ---
--- Running Fundamentals Agent ---
[Fundamentals Agent] Calling get_company_overview with {'ticker': 'MU'}
[Fundamentals Agent] Calling get_company_overview with {'ticker': 'AMAT'}
[Fundamentals Agent] Calling get_company_overview with {'ticker': 'LRCX'}
--- Critic Evaluating Fundamentals Agent ---
--- Running Sentiment

---
## 🛠️ Task 5 — Implement the LLM Evaluator (15 pts)

The evaluator is an **LLM-as-judge**: it reads a question, the expected answer description, and an agent's actual answer, then scores it.

This is how you measure accuracy across all architectures automatically — rather than reading 45 answers by hand.

---

### Required output format

Your evaluator must return a Python dict with exactly these keys.  
The evaluation runner reads all of them to fill the Excel sheet:

```python
{
    "score"                 : int,        # 0, 1, 2, or 3
    "max_score"             : int,        # always 3
    "reasoning"             : str,        # one sentence explaining the score
    "hallucination_detected": bool,       # True if the answer contains invented facts
    "key_issues"            : list[str],  # specific problems found; empty list if none
}
```

---

### Scoring rubric to include in your prompt

```
3 — Fully correct:    all required data present, numbers accurate, conditions met
2 — Partially correct: key data present but incomplete, gaps, or minor inaccuracies
1 — Mostly wrong:     attempted but wrong numbers, missed required conditions,
                      or claims that appear fabricated
0 — Complete failure: refused to answer, said data unavailable without trying tools,
                      or answer has no relevance to the question
```

---

### Hallucination detection — what to flag

Include these rules in your prompt:
- Specific numbers (prices, P/E ratios, % changes) with no tool data to support them
- Stock tickers that don't exist or aren't relevant to the question
- Definitive claims about "current" data without having called a live data tool

---

### Important considerations

**The evaluator only sees text** — it cannot verify numbers against live data.  
Its hallucination detection is based on reasoning about whether claims look fabricated,  
not on cross-checking against a ground truth source.  
You will reflect on this limitation in the graded questions.

**JSON parsing:** The LLM may wrap its response in markdown fences (```json ... ```).  
Strip those before parsing. Return the fallback dict on any parse error.

**Prompt placement:** Pass the rubric, the expected answer description, and the agent's  
actual answer all in one user message so the evaluator has full context.

---

### Calibration tests (provided below)

Three test cases check your evaluator before the full run:
1. A clearly correct answer (expect score = 3)
2. An answer with an invented number (expect `hallucination_detected = True`, score ≤ 1)
3. A refusal (expect score = 0)

All three must behave as expected before running the full evaluation.


In [ ]:
# ── YOUR EVALUATOR IMPLEMENTATION ────────────────────────────
#
# Things to decide:
#   - How detailed is your rubric? (more detail → more consistent scores)
#   - How do you instruct it to detect hallucinations vs honest uncertainty?
#   - How strict are you on partial answers? (affects SA vs MA comparison)
#   - How do you handle "I don't know" answers — 0 or 1?
#
# Required: function signature must be exactly as shown below.

# ==========================================
# LLM EVALUATOR (LLM-AS-JUDGE)
# ==========================================

EVALUATOR_SYSTEM_PROMPT = """
You are an expert LLM-as-judge evaluator for a financial AI assistant.
Your job is to read a question, the expected answer description, and an agent's actual answer, then score it.
You MUST output your evaluation strictly as a valid JSON object matching this structure exactly:
{
    "score"                 : 0,
    "max_score"             : 3,
    "reasoning"             : "one sentence explaining the score",
    "hallucination_detected": false,
    "key_issues"            : []
}
IMPORTANT: Return ONLY raw JSON, with no markdown fences, prefixes, or suffixes.
"""

def run_evaluator(question: str, expected_answer: str, agent_answer: str) -> dict:
    import json

    fallback = {
        "score": 0, "max_score": 3, "reasoning": "evaluator parse error",
        "hallucination_detected": False, "key_issues": ["evaluator failed to parse"]
    }

    rubric = """
### Scoring Rubric:
3 — Fully correct:    all required data present, numbers accurate, conditions met
2 — Partially correct: key data present but incomplete, gaps, or minor inaccuracies
1 — Mostly wrong:     attempted but wrong numbers, missed required conditions,
                      or claims that appear fabricated
0 — Complete failure: refused to answer, said data unavailable without trying tools,
                      or answer has no relevance to the question

### Hallucination Detection Rules:
NOTE: You can only see the agent's final text answer, not its tool execution traces. You MUST ASSUME the agent successfully called tools if it confidently provides the requested specific numbers, unless stated otherwise.
Flag hallucination_detected as `true` if:
- The agent explicitly states it is guessing, estimating, or doesn't have access to the data, but provides a specific number anyway.
- The answer provides specific numeric data (e.g., P/E ratio, prices, percentages) with qualifiers like "approximately", "around", or "based on current market conditions" without explicitly referencing a tool or source, especially when the expected answer implies fetching a precise value from a tool.
- Stock tickers that don't exist or aren't relevant to the question.
- The answer clearly contradicts the expected answer type or invents unrelated information.
"""

    content = f"{rubric}\n\nQuestion: {question}\n\nExpected Answer: {expected_answer}\n\nAgent's Actual Answer: {agent_answer}"

    try:
        response = get_chat_completion([
            {"role": "system", "content": EVALUATOR_SYSTEM_PROMPT},
            {"role": "user", "content": content}
        ])

        res_text = response.choices[0].message.content.strip()

        if res_text.startswith("```json"):
            res_text = res_text[7:-3].strip()
        elif res_text.startswith("```"):
            res_text = res_text[3:-3].strip()

        parsed = json.loads(res_text)

        # Ensure all required keys exist and have appropriate types
        return {
            "score": int(parsed.get("score", 0)),
            "max_score": 3,
            "reasoning": str(parsed.get("reasoning", "")),
            "hallucination_detected": bool(parsed.get("hallucination_detected", False)),
            "key_issues": list(parsed.get("key_issues", []))
        }
    except Exception as e:
        print(f"Evaluator error: {e}")
        return fallback

In [ ]:
# ── Calibration tests — all three must behave as expected ─────
print("=== Calibration Test 1 — correct answer (expect score=3) ===")
t1 = run_evaluator(
    question        = "What is the P/E ratio of Apple (AAPL)?",
    expected_answer = "Should return AAPL P/E ratio as a single numeric value from Alpha Vantage.",
    agent_answer    = "The current P/E ratio of Apple Inc. (AAPL) is 33.45.",
)
print(f"  Score: {t1['score']}/3 | Hallucination: {t1['hallucination_detected']}")
print(f"  Reasoning: {t1['reasoning']}")

print("\n=== Calibration Test 2 — fabricated number (expect hallucination=True, score≤1) ===")
t2 = run_evaluator(
    question        = "What is the P/E ratio of Apple (AAPL)?",
    expected_answer = "Should return AAPL P/E ratio as a single numeric value from Alpha Vantage.",
    agent_answer    = "Apple's P/E ratio is approximately 28.5 based on current market conditions.",
)
print(f"  Score: {t2['score']}/3 | Hallucination: {t2['hallucination_detected']}")
print(f"  Reasoning: {t2['reasoning']}")
assert t2["hallucination_detected"] == True, "Should detect fabricated P/E as hallucination"

print("\n=== Calibration Test 3 — refusal (expect score=0) ===")
t3 = run_evaluator(
    question        = "What is the P/E ratio of Apple (AAPL)?",
    expected_answer = "Should return AAPL P/E ratio as a single numeric value from Alpha Vantage.",
    agent_answer    = "I cannot retrieve real-time financial data. Please check Yahoo Finance.",
)
print(f"  Score: {t3['score']}/3 | Hallucination: {t3['hallucination_detected']}")
print(f"  Reasoning: {t3['reasoning']}")
assert t3["score"] == 0, "Refusal should score 0"

print("\n✅ Evaluator calibration complete")

=== Calibration Test 1 — correct answer (expect score=3) ===
  Score: 3/3 | Hallucination: False
  Reasoning: The answer provides the correct P/E ratio for Apple Inc. (AAPL) as a single numeric value.

=== Calibration Test 2 — fabricated number (expect hallucination=True, score≤1) ===
  Score: 1/3 | Hallucination: True
  Reasoning: The answer provides a P/E ratio but includes qualifiers like 'approximately' and 'based on current market conditions', indicating a lack of precision and reliance on estimation.

=== Calibration Test 3 — refusal (expect score=0) ===
  Score: 0/3 | Hallucination: False
  Reasoning: The agent refused to provide the requested P/E ratio and directed the user to another source, failing to attempt to retrieve the data.

✅ Evaluator calibration complete


## Step 5 — Benchmark Questions (Fixed — Do Not Modify)

15 questions across three difficulty levels. All three architectures run against all 15.

| Tier | Questions | What makes them harder |
|---|---|---|
| Easy (Q01–Q05) | 5 | 1 tool, single domain |
| Medium (Q06–Q10) | 5 | 2 tools, cross-domain reasoning |
| Hard (Q11–Q15) | 5 | 3+ tools, multi-condition filtering or cross-domain synthesis |


In [ ]:
BENCHMARK_QUESTIONS = [
    # ── EASY ──────────────────────────────────────────────────────────────
    {"id":"Q01","complexity":"easy","category":"sector_lookup",
     "question":"List all semiconductor companies in the database.",
     "expected":"Should return company names and tickers for semiconductor stocks from the local DB. "
                "Tickers include NVDA, AMD, INTC, QCOM, AVGO, TXN, ADI, MU and others."},
    {"id":"Q02","complexity":"easy","category":"market_status",
     "question":"Are the US stock markets open right now?",
     "expected":"Should return the current open/closed status for NYSE and NASDAQ "
                "with their trading hours."},
    {"id":"Q03","complexity":"easy","category":"fundamentals",
     "question":"What is the P/E ratio of Apple (AAPL)?",
     "expected":"Should return AAPL P/E ratio as a single numeric value fetched from Alpha Vantage."},
    {"id":"Q04","complexity":"easy","category":"sentiment",
     "question":"What is the latest news sentiment for Microsoft (MSFT)?",
     "expected":"Should return 3–5 recent MSFT headlines with Bullish/Bearish/Neutral labels and scores."},
    {"id":"Q05","complexity":"easy","category":"price",
     "question":"What is NVIDIA's stock price performance over the last month?",
     "expected":"Should return NVDA start price, end price, and % change for the 1-month period."},

    # ── MEDIUM ─────────────────────────────────────────────────────────────
    {"id":"Q06","complexity":"medium","category":"price_comparison",
     "question":"Compare the 1-year price performance of AAPL, MSFT, and GOOGL. Which grew the most?",
     "expected":"Should fetch 1y performance for all 3 tickers, return % change for each, "
                "and identify the highest performer."},
    {"id":"Q07","complexity":"medium","category":"fundamentals",
     "question":"Compare the P/E ratios of AAPL, MSFT, and NVDA. Which looks most expensive?",
     "expected":"Should return P/E ratios for all 3 tickers and identify which has the highest P/E."},
    {"id":"Q08","complexity":"medium","category":"sector_price",
     "question":"Which energy stocks in the database had the best 6-month performance?",
     "expected":"Should query the DB for energy sector tickers, fetch 6-month price performance "
                "for each, and return them ranked by % change."},
    {"id":"Q09","complexity":"medium","category":"sentiment",
     "question":"What is the news sentiment for Tesla (TSLA) and how has its stock moved this month?",
     "expected":"Should return TSLA news sentiment (label + score) AND 1-month price % change "
                "from two separate tool calls."},
    {"id":"Q10","complexity":"medium","category":"fundamentals",
     "question":"What are the 52-week high and low for JPMorgan (JPM) and Goldman Sachs (GS)?",
     "expected":"Should return 52-week high and low for both JPM and GS fetched from Alpha Vantage."},

    # ── HARD ───────────────────────────────────────────────────────────────
    {"id":"Q11","complexity":"hard","category":"multi_condition",
     "question":"Which tech stocks dropped this month but grew this year? Return the top 3.",
     "expected":"Should get tech tickers from DB, fetch both 1-month and year-to-date performance, "
                "filter for negative 1-month AND positive YTD, return top 3 by yearly growth with "
                "exact percentages. Results must satisfy both conditions simultaneously."},
    {"id":"Q12","complexity":"hard","category":"multi_condition",
     "question":"Which large-cap technology stocks on NASDAQ have grown more than 20% this year?",
     "expected":"Should query DB for large-cap NASDAQ tech stocks, fetch YTD performance, "
                "filter for >20% growth, and return matching tickers with exact % change."},
    {"id":"Q13","complexity":"hard","category":"cross_domain",
     "question":"For the top 3 semiconductor stocks by 1-year return, what are their P/E ratios "
                "and current news sentiment?",
     "expected":"Should find semiconductor tickers in DB, rank by 1-year return to find top 3, "
                "then fetch P/E ratio AND news sentiment for each — requiring three separate "
                "data domains (price, fundamentals, sentiment)."},
    {"id":"Q14","complexity":"hard","category":"cross_domain",
     "question":"Compare the market cap, P/E ratio, and 1-year stock performance of JPM, GS, and BAC.",
     "expected":"Should return market cap, P/E, and 1-year % change for all 3 tickers, "
                "combining Alpha Vantage fundamentals and yfinance price data."},
    {"id":"Q15","complexity":"hard","category":"multi_condition",
     "question":"Which finance sector stocks are trading closer to their 52-week low than their "
                "52-week high? Return the news sentiment for each.",
     "expected":"Should get finance sector tickers from DB, fetch 52-week high and low for each, "
                "compute proximity to the low, then fetch news sentiment for qualifying stocks."},
]

print(f"✅ {len(BENCHMARK_QUESTIONS)} questions loaded")
for tier in ["easy","medium","hard"]:
    count = sum(1 for q in BENCHMARK_QUESTIONS if q["complexity"]==tier)
    print(f"   {tier:8s}: {count} questions")

✅ 15 questions loaded
   easy    : 5 questions
   medium  : 5 questions
   hard    : 5 questions


## Step 6 — Evaluation Runner and Excel Writer (Provided)

This runner calls all three architectures on all 15 questions, evaluates each answer,  
and writes results to an Excel file with two sheets:

- **Results** — one row per question with all metrics for all three architectures  
- **Summary** — accuracy % by architecture and difficulty tier (auto-calculated)

Results are saved after every question — if the run crashes, you do not lose progress.

### Excel columns produced

| Column group | Columns written |
|---|---|
| Question | ID, complexity, category, question text, expected answer |
| Baseline | answer, time(s), eval score (0-3), eval reasoning, hallucination detected, issues |
| Single Agent | answer, tools used, tool count, iterations, time(s), eval score, reasoning, hallucination, issues |
| Multi Agent | answer, tools used, tool count, time(s), confidence, critic issues, agents activated, architecture, eval score, reasoning, hallucination, issues |


In [ ]:
@dataclass
class EvalRecord:
    # Question
    question_id : str;  question    : str;  complexity : str
    category    : str;  expected    : str
    # Baseline
    bl_answer       : str   = "";  bl_time         : float = 0.0
    bl_score        : int   = -1;  bl_reasoning    : str   = ""
    bl_hallucination: str   = "";  bl_issues       : str   = ""
    # Single agent
    sa_answer       : str   = "";  sa_tools        : str   = ""
    sa_tool_count   : int   = 0;   sa_iters        : int   = 0
    sa_time         : float = 0.0; sa_score        : int   = -1
    sa_reasoning    : str   = "";  sa_hallucination: str   = ""
    sa_issues       : str   = ""
    # Multi agent
    ma_answer       : str   = "";  ma_tools        : str   = ""
    ma_tool_count   : int   = 0;   ma_time         : float = 0.0
    ma_confidence   : str   = "";  ma_critic_issues: int   = 0
    ma_agents       : str   = "";  ma_architecture : str   = ""
    ma_score        : int   = -1;  ma_reasoning    : str   = ""
    ma_hallucination: str   = "";  ma_issues       : str   = ""


# ── Column rename map (internal name → Excel header) ─────────
_COL_NAMES = {
    "question_id":"Question ID","question":"Question","complexity":"Difficulty",
    "category":"Category","expected":"Expected Answer",
    "bl_answer":"Baseline Answer","bl_time":"Baseline Time (s)",
    "bl_score":"Baseline Score /3","bl_reasoning":"Baseline Eval Reasoning",
    "bl_hallucination":"Baseline Hallucination","bl_issues":"Baseline Issues",
    "sa_answer":"SA Answer","sa_tools":"SA Tools Used","sa_tool_count":"SA Tool Count",
    "sa_iters":"SA Iterations","sa_time":"SA Time (s)",
    "sa_score":"SA Score /3","sa_reasoning":"SA Eval Reasoning",
    "sa_hallucination":"SA Hallucination","sa_issues":"SA Issues",
    "ma_answer":"MA Answer","ma_tools":"MA Tools Used","ma_tool_count":"MA Tool Count",
    "ma_time":"MA Time (s)","ma_confidence":"MA Avg Confidence",
    "ma_critic_issues":"MA Critic Issue Count","ma_agents":"MA Agents Activated",
    "ma_architecture":"MA Architecture",
    "ma_score":"MA Score /3","ma_reasoning":"MA Eval Reasoning",
    "ma_hallucination":"MA Hallucination","ma_issues":"MA Issues",
}


def _save_excel(records: list, path: str):
    df = pd.DataFrame([r.__dict__ for r in records]).rename(columns=_COL_NAMES)

    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        # ── Sheet 1: full results ──────────────────────────────
        df.to_excel(writer, index=False, sheet_name="Results")

        # ── Sheet 2: summary by architecture × difficulty ──────
        rows = []
        for arch, sc, tc, hc in [
            ("Baseline",     "Baseline Score /3", "Baseline Time (s)", "Baseline Hallucination"),
            ("Single Agent", "SA Score /3",       "SA Time (s)",       "SA Hallucination"),
            ("Multi Agent",  "MA Score /3",       "MA Time (s)",       "MA Hallucination"),
        ]:
            for tier in ["easy", "medium", "hard", "all"]:
                subset = df if tier == "all" else df[df["Difficulty"] == tier]
                valid  = subset[subset[sc] >= 0]
                avg_s  = valid[sc].mean() if len(valid) else 0
                rows.append({
                    "Architecture"   : arch,
                    "Difficulty"     : tier,
                    "Questions Scored": len(valid),
                    "Avg Score /3"   : round(avg_s, 2),
                    "Accuracy %"     : round(avg_s / 3 * 100, 1),
                    "Avg Time (s)"   : round(df[tc].mean(), 1),
                    "Hallucinations" : (df[hc] == "True").sum(),
                })
        pd.DataFrame(rows).to_excel(writer, index=False, sheet_name="Summary")


def run_full_evaluation(output_xlsx: str = "results.xlsx", delay_sec: float = 3.0):
    """
    Run all 15 questions through baseline, single agent, and multi agent.
    Score each answer. Write results to Excel after every question.
    """
    records = []
    total   = len(BENCHMARK_QUESTIONS)
    print(f"\n{'='*62}")
    print(f"  FULL EVALUATION  |  {total} questions × 3 architectures")
    print(f"  Model: {ACTIVE_MODEL}  |  Output: {output_xlsx}")
    print(f"{'='*62}\n")

    for i, q in enumerate(BENCHMARK_QUESTIONS, 1):
        print(f"[{i:02d}/{total}] {q['id']} ({q['complexity']:6s}) {q['question'][:52]}...")
        rec = EvalRecord(question_id=q["id"], question=q["question"],
                         complexity=q["complexity"], category=q["category"],
                         expected=q["expected"])

        # ── Baseline ───────────────────────────────────────────
        print("         baseline  ...", end=" ", flush=True)
        try:
            t0 = time.time()
            bl = run_baseline(q["question"], verbose=False)
            rec.bl_answer = bl.answer.replace("\n", " ")
            rec.bl_time   = round(time.time() - t0, 2)
            ev = run_evaluator(q["question"], q["expected"], bl.answer)
            rec.bl_score        = ev.get("score", -1)
            rec.bl_reasoning    = ev.get("reasoning", "")
            rec.bl_hallucination= str(ev.get("hallucination_detected", False))
            rec.bl_issues       = " | ".join(ev.get("key_issues", []))
            print(f"✅  {rec.bl_time:5.1f}s  score {rec.bl_score}/3")
        except Exception as e:
            print(f"❌  {e}")

        # ── Single Agent ───────────────────────────────────────
        print("         single    ...", end=" ", flush=True)
        try:
            t0 = time.time()
            sa = run_single_agent(q["question"], verbose=False)
            rec.sa_answer    = sa.answer.replace("\n", " ")
            rec.sa_tools     = ", ".join(sa.tools_called)
            rec.sa_tool_count= len(sa.tools_called)
            rec.sa_iters     = len(sa.tools_called) + 1   # approx
            rec.sa_time      = round(time.time() - t0, 2)
            ev = run_evaluator(q["question"], q["expected"], sa.answer)
            rec.sa_score        = ev.get("score", -1)
            rec.sa_reasoning    = ev.get("reasoning", "")
            rec.sa_hallucination= str(ev.get("hallucination_detected", False))
            rec.sa_issues       = " | ".join(ev.get("key_issues", []))
            print(f"✅  {rec.sa_time:5.1f}s  score {rec.sa_score}/3"
                  f"  tools [{rec.sa_tools or 'none'}]")
        except Exception as e:
            print(f"❌  {e}")

        # ── Multi Agent ────────────────────────────────────────
        print("         multi     ...", end=" ", flush=True)
        try:
            t0  = time.time()
            ma  = run_multi_agent(q["question"], verbose=False)
            res = ma.get("agent_results", [])
            all_tools  = [t for r in res for t in r.tools_called]
            all_issues = [iss for r in res for iss in r.issues_found]
            avg_conf   = sum(r.confidence for r in res) / len(res) if res else 0.0
            rec.ma_answer        = ma["final_answer"].replace("\n", " ")
            rec.ma_tools         = ", ".join(dict.fromkeys(all_tools))
            rec.ma_tool_count    = len(all_tools)
            rec.ma_time          = round(time.time() - t0, 2)
            rec.ma_confidence    = f"{avg_conf:.0%}"
            rec.ma_critic_issues = len(all_issues)
            rec.ma_agents        = ", ".join(r.agent_name for r in res)
            rec.ma_architecture  = ma.get("architecture", "")
            ev = run_evaluator(q["question"], q["expected"], ma["final_answer"])
            rec.ma_score        = ev.get("score", -1)
            rec.ma_reasoning    = ev.get("reasoning", "")
            rec.ma_hallucination= str(ev.get("hallucination_detected", False))
            rec.ma_issues       = " | ".join(ev.get("key_issues", []))
            print(f"✅  {rec.ma_time:5.1f}s  score {rec.ma_score}/3"
                  f"  conf {rec.ma_confidence}  issues {rec.ma_critic_issues}")
        except Exception as e:
            print(f"❌  {e}")

        records.append(rec)
        _save_excel(records, output_xlsx)       # save progress after every question

        if i < total:
            print(f"         ⏳ waiting {delay_sec}s ...\n")
            time.sleep(delay_sec)

    # ── Print summary table ────────────────────────────────────
    print(f"\n{'='*62}  RESULTS")
    print(f"{'Architecture':<18} {'Easy':>8} {'Medium':>8} {'Hard':>8} {'Overall':>8}")
    print("─" * 52)
    for arch, sk in [("Baseline","bl_score"),("Single Agent","sa_score"),("Multi Agent","ma_score")]:
        def pct(tier):
            s = [getattr(r, sk) for r in records
                 if getattr(r, sk) >= 0 and (tier=="all" or r.complexity==tier)]
            return f"{sum(s)/len(s)/3*100:.0f}%" if s else "—"
        print(f"{arch:<18} {pct('easy'):>8} {pct('medium'):>8} {pct('hard'):>8} {pct('all'):>8}")

    print(f"\n✅ Saved → {output_xlsx}")
    return output_xlsx

print("✅ Evaluation runner ready")

✅ Evaluation runner ready


## Step 7 — Run the Evaluation

Run the sanity check cell first. Only proceed to the full evaluation once all three architectures  
produce valid output on the test question.


In [ ]:
# ── Sanity check — one question, all three architectures ──────
q_test = BENCHMARK_QUESTIONS[2]        # Q03 — easy fundamentals
print(f"Test question: {q_test['question']}\n")

bl_t = run_baseline(q_test["question"], verbose=False)
sa_t = run_single_agent(q_test["question"], verbose=False)
ma_t = run_multi_agent(q_test["question"], verbose=False)

print(f"Baseline     : {bl_t.answer[:120]}")
print(f"Single Agent : {sa_t.answer[:120]}  |  tools: {sa_t.tools_called}")
print(f"Multi Agent  : {ma_t['final_answer'][:120]}  |  arch: {ma_t['architecture']}")

ev_bl = run_evaluator(q_test["question"], q_test["expected"], bl_t.answer)
ev_sa = run_evaluator(q_test["question"], q_test["expected"], sa_t.answer)
ev_ma = run_evaluator(q_test["question"], q_test["expected"], ma_t["final_answer"])
print(f"\nScores — Baseline: {ev_bl['score']}/3  |  Single: {ev_sa['score']}/3  |  Multi: {ev_ma['score']}/3")

Test question: What is the P/E ratio of Apple (AAPL)?

Baseline     : As of my last knowledge update in October 2023, the P/E ratio of Apple (AAPL) was approximately in the range of 25 to 30
Single Agent : The P/E ratio of Apple Inc (AAPL) is 32.37.  |  tools: ['get_company_overview']
Multi Agent  : The P/E ratio for Apple Inc. (AAPL) is 32.37.  |  arch: orchestrator-critic

Scores — Baseline: 1/3  |  Single: 3/3  |  Multi: 3/3


# Full Evaluation was run on a separate script. To avoid wasting resources, they were not re-run in this notebook

In [ ]:
# ── Full evaluation — gpt-4o-mini ─────────────────────────────
ACTIVE_MODEL = MODEL_SMALL
run_full_evaluation(output_xlsx="results_gpt4o_mini.xlsx", delay_sec=3.0)

In [ ]:
# ── Full evaluation — gpt-4o (required for reflection Q4) ─────
ACTIVE_MODEL = MODEL_LARGE
run_full_evaluation(output_xlsx="results_gpt4o.xlsx", delay_sec=3.0)

---
## 📝 Graded Reflection Questions (30 pts)

Answer each question in the markdown cell below it.  
Every answer must reference specific question IDs and scores from your Excel results.  
Vague answers without data will receive at most half marks.


### Q0 — Compare the performance of baseline vs single agent implementation? (5 pts)
- Analyse whether the usecase needs the agentic implementation, if yes why? if no, why not?


The Baseline Agent (a standard gpt-4o-mini call) fails to answer real-time financial questions. Without tools to fetch live data, it cannot access current stock prices, P/E ratios, or news sentiment. When asked for specific metrics like Apple's P/E ratio, it refuses to answer or hallucinates outdated numbers from its static training data. <br>
In contrast, the Single Agent dramatically outperforms the Baseline by utilizing external API and database tools. Crucially, it performs multi-step reasoning, such as querying a database for semiconductor tickers, iterating through them to pull live price data, and compiling a factually accurate answer. <br>
The agentic implementation is strictly required for this use case. The financial domain relies on rapidly changing, real-time data. A static LLM cannot track live market fluctuations or breaking news. An orchestration engine is necessary to actively retrieve data, compare metrics, and prevent dangerous hallucinations in a domain where accuracy is paramount. <br>
For example, asking gpt-4o-mini what the current stock price will yield an invalid result as it is unable to pull any current data. (e.g. benchmark question 3)

### Q1 — Is Multi-Agent Actually Necessary? (5 pts)

Using your `results_gpt4o_mini.xlsx`:

- For which difficulty tier (easy / medium / hard) does multi-agent most outperform single agent?
- For which tier is the difference smallest?
- Give **2 concrete examples** — one where multi-agent clearly won, one where single agent was equivalent or better.  
  For each example, cite the question ID, both scores, and explain *why the architecture made the difference*.


In theory, a Multi-Agent architecture will outperform a Single Agent approach due to three key advantages: <br>


1.   Specialization: By dividing the 7 tools among specialist agents (e.g., a Price Agent
and a DB Agent), each LLM's system prompt is narrow and focused. This
significantly reduces tool-calling errors and context confusion compared to a
heavily loaded single prompt.
2.   Built-in Verification: Multi-agent systems use an Orchestrator or Critic agent to
review responses. If a specialist provides an incomplete answer, the Orchestrator
can force a redo, drastically reducing hallucinations.

3.   Complex Synthesis: For multi-condition queries, an Orchestrator can parallelize
work (e.g., fetching P/E and sentiment simultaneously) and synthesize the results
cleanly, avoiding the infinite loops or context limits that often break Single Agents.


For Question 1, it looks like single agent performed just as well as multiagent. In this scenario, the single agent did not have to call multiple tools, and the task was relatively simple.
<br>
For Question 13, the multi-agent provided real data while the single agent hallucinated an answer.

Multi-agent performs the best in hard questions.




### Q2 — Is the Evaluator Reliable? (5 pts)

Find **3 questions** in your results where you disagree with the score the evaluator assigned.  
For each one:
- What score did it give, and what score do you think is correct?
- Why did it get it wrong — did it miss a hallucination, or penalise an answer that was actually correct?

Then answer: is your evaluator systematically biased in any direction?  
What would you change in your prompt to fix the biggest problem you found?


The evaluator is unreliable because it does not have access to any ground truth data. It is only evaluating whether an answer LOOKS like a hallucination or making up data. In practice, we would want an evaluator to have access to real time data to accurately judge the correctness of an answer. <br>
As it stands, the evaluator implementation of the LLM is just looking for signs that an answer was fabricated, avoided the question, or included extraneous data. It does not provide any truth to the provided answer. The agentic models presented here could be making up all information and the evaluator would be none the wiser because it doesn't have any real-time information.
<br>
Three questions that it incorrectly scored:
Single Agent MSFT News Sentiment - Given 0, should be 3 because it provided a correct negative answer.
Multi-Agent News sentiment - Given 0, should be 3 because it provided a negative answer when there was no news.
Question 10 Single Agent - Given a 2, should be 3. A negative response to news sentiment shouldn't be penalized
<br>
The trend is that evaluator is labeling no data as missing data. I would add to the evaluator's instructions that no data or sentiment is an acceptable answer.

### Q3 — Accuracy Across Architectures and Difficulty Tiers (5 pts)

Fill in the table below from your `results_gpt4o_mini.xlsx` Summary sheet, then write your analysis.

| Architecture | Easy % | Medium % | Hard % | Overall % |
|---|---|---|---|---|
| Baseline |20% |6.7% |6.7% |11.1% |
| Single Agent |33.3% |53.3% |26.7% |37.8% |
| Multi Agent |33.3% |46.7% |6.7% |28.9% |

- Where does each architecture most significantly break down?
- Is the accuracy drop from medium → hard larger for some architectures than others?
- What does this tell you about *which type of question* benefits most from an agentic approach?

Baseline architecture struggles all throughout this exercise due to its lack of up to date data.
Single agent does the best overall due to its robust tendency to not overcomplicate the question or second guess itself.
THe multi-agent response has too much back-and-forth between the orchestrator and the critic that rejects information or deems some answers unacceptable when in fact they are.


### Q4 — gpt-4o-mini vs gpt-4o with Your Multi-Agent (5 pts)

Compare `results_gpt4o_mini.xlsx` and `results_gpt4o.xlsx` for the **multi-agent architecture only**.

- Which question categories show the biggest accuracy improvement with the larger model?
- Does the confidence score (or critic issue count) change meaningfully?
- On hard questions specifically — is the accuracy difference large enough to justify the cost?
- Is there any category where the smaller model is actually sufficient?


Multi-agent performed significantly better in the GPT-4o vs GPT-4o-mini, 44.4% vs 28.9% overall. Hard question accuracy jumped from 6.7% up to 46.7%, showing the largest improvement. This result demonstrates that the larger, more complex model was better able to deal with complex questions and multiple tools.
<br>
On hard questions, the accuracy difference is large enough to justify the cost with the ultimate task of finding financial data. For stock traders, investors, or hedge funds, up-to-date and accurate data are priceless.
<br>
For simple questions, the single agent with the smaller model is sufficient, dmonstrating the same accuracy overall at 37.8%.
<br>
The overall results are not re-assuring for making any decisions, but some general observatons likely hold true, no matter the actual results.


### Q5 — Your Multi-Agent Design Decisions (5 pts)

Document the architecture you built:

1. Which pattern did you choose (orchestrator-critic, pipeline, parallel, or hybrid)? Why?
2. How did you divide tools between specialists? Explain each agent's scope.
3. What is your verification / confidence mechanism and how does it work?
4. What did you try first that did not work, and what did you change?
5. Looking at your results — did your architecture actually reduce hallucinations compared to the single agent? Show the numbers.



We chose the orchestrator-critic pattern. This pattern works best because financial questions frequently require sequential data retrieval across multiple domains (e.g., you often need to fetch tickers from the database first, before fetching their prices or P/E ratios). The orchestrator effectively plans these steps sequentially. The critic component was added directly after each specialist agent's execution to ensure their numerical outputs matched the raw JSON from the tool, anchoring the outputs firmly to reality.

Database Agent: Handles stock discovery by sector and industry via local SQLite queries (get_tickers_by_sector, query_local_db).
Fundamentals Agent: Retrieves static and point-in-time company metrics like P/E ratio, EPS, and 52-week highs/lows (get_company_overview).
Technical/Price Agent: Focuses on historical stock price performance and checks if the market is open (get_price_performance, get_market_status).
Sentiment Agent: Fetches recent news headlines and provides aggregated sentiment scores (get_news_sentiment).

The verification mechanism is handled by the Critic Agent.

How it works: After a specialist agent completes a task, the Critic receives both the agent's textual answer and the raw JSON tool data. It verifies that every number, price, or text claim matches the actual JSON output.
It then returns a strict JSON object containing a confidence score (ranging from 0.0 to 1.0) and lists any issues_found. The Synthesizer then uses this reviewed information and confidence value to construct a safe final answer.

Initially, we tried allowing all agents to execute simultaneously (a Parallel pattern) or creating a single massive prompt. This struggled because complex cross-domain questions (like "find the P/E of top tech stocks") natively break when agents don't have the context of the tickers found by earlier tools. We changed the design to an Orchestrator that creates a sequential step-by-step JSON plan. Additionally, we found JSON extraction from LLMs sometimes failed, so we added fallback parsing (stripping markdown fences) to make the orchestrator more robust.

Looking at the rigorous GPT-4o evaluation results, both tool-augmented agents eliminated hallucinations, reducing them from 3 (in the baseline) to 0. While the Multi-Agent architecture resulted in the same 0 hallucinations as the Single Agent, it vastly improved reasoning and execution accuracy on complex multi-step problems. For GPT-4o, the Multi-Agent architecture improved the score on hard questions dramatically compared to the Single Agent:

Baseline Hallucinations: 3 (across all difficulties)
Single Agent Hallucinations: 0 | Hard Score (GPT-4o): 0.60/3.00 (20.0%)
Multi-Agent Hallucinations: 0 | Hard Score (GPT-4o): 1.40/3.00 (46.7%)
The orchestrator-critic setup secured factual grounding while significantly boosting complex problem-solving ability over a single agent model.



---
## ✅ Submission Checklist

- [X] `get_company_overview()` — all assertions in Task 1 pass
- [X] `get_tickers_by_sector()` — sector match AND industry fallback working
- [X] `run_baseline()` — `tools_called == []`, answer not empty
- [X] `run_single_agent()` — uses tools, system prompt reasoning documented in comments
- [X] `run_multi_agent()` — returns correct dict contract, architecture documented in comments
- [X] `run_evaluator()` — all 3 calibration tests pass
- [X] `results_gpt4o_mini.xlsx` — Results + Summary sheets filled for all 15 questions
- [X] `results_gpt4o.xlsx` — Results + Summary sheets filled for all 15 questions
- [X] All 5 reflection questions answered with question IDs and scores cited

**Submit:** this notebook + `results_gpt4o_mini.xlsx` + `results_gpt4o.xlsx` + insights.doc/pdf (explaning design choices)
